In [ ]:
import numpy as np
import random

# 第17章 SAC算法

## 目录
1. SAC原理
2. 最大熵框架
3. 编程题

---
## 1. SAC原理

In [ ]:
"""SAC (Soft Actor-Critic)"""
class SAC:
    """SAC算法"""
    def __init__(self, n_states, n_actions, alpha=0.2):
        self.actor_mean = np.zeros((n_states, n_actions))
        self.actor_log_std = np.zeros((n_states, n_actions))
        self.critic1 = np.zeros((n_states, n_actions))
        self.critic2 = np.zeros((n_states, n_actions))
        self.alpha = alpha  # 熵温度
        self.target_entropy = -n_actions
    
    def get_action(self, state):
        std = np.exp(self.actor_log_std[0])
        action = self.actor_mean[0] + np.random.randn() * std
        return np.tanh(action)
    
    def entropy(self, state):
        return self.alpha * np.log(np.exp(self.actor_log_std[0]) + 1e-8)

print("SAC: 最大熵+双Q网络+策略网络")

---
## 2. 最大熵框架

In [ ]:
def soft_value(state, q, entropy):
    """软价值函数"""
    return q + entropy

def soft_update(target, source, tau=0.005):
    """软更新"""
    return target + tau * (source - target)

class ReplayBuffer:
    """经验回放"""
    def __init__(self, capacity=10000):
        self.buffer = []
        self.capacity = capacity
    def push(self, transition):
        if len(self.buffer) >= self.capacity: self.buffer.pop(0)
        self.buffer.append(transition)
    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

print("最大熵: 鼓励探索+稳定训练")

---
## 3. 编程题

### 编程题1：实现SAC算法在连续控制环境中的训练

In [ ]:
class SACAgent:
    """SAC智能体"""
    def __init__(self, n_states, n_actions, alpha=0.2):
        self.actor_mean = np.zeros(n_actions)
        self.actor_log_std = np.zeros(n_actions)
        self.critic1 = np.zeros(n_actions)
        self.critic2 = np.zeros(n_actions)
        self.target_critic1 = self.critic1.copy()
        self.target_critic2 = self.critic2.copy()
        self.alpha = alpha
        self.gamma = 0.99
    
    def get_action(self, state, deterministic=False):
        if deterministic:
            return np.tanh(self.actor_mean)
        std = np.exp(self.actor_log_std)
        action = self.actor_mean + np.random.randn() * std
        return np.tanh(action)
    
    def update(self, s, a, r, ns, done, alpha_a=0.001, alpha_c=0.01):
        target = r + self.gamma * (0 if done else min(np.dot(ns, self.target_critic1), np.dot(ns, self.target_critic2)))
        q1_loss = (target - np.dot(s, self.critic1)) ** 2
        q2_loss = (target - np.dot(s, self.critic2)) ** 2
        self.critic1 -= alpha_c * q1_loss * s
        self.critic2 -= alpha_c * q2_loss * s
        
        a_loss = -self.critic1 - self.alpha * np.exp(self.actor_log_std)
        self.actor_mean -= alpha_a * a_loss
        
        self.target_critic1 += 0.005 * (self.critic1 - self.target_critic1)
        self.target_critic2 += 0.005 * (self.critic2 - self.target_critic2)

class ContinuousEnv:
    """连续环境"""
    def __init__(self):
        self.state = np.random.randn(2) * 0.1
    def reset(self): self.state = np.random.randn(2) * 0.1; return self.state.copy()
    def step(self, action):
        action = np.clip(action[0], -1, 1) * 2.0
        self.state += np.array([self.state[1], -self.state[0] + action]) * 0.05
        reward = -(self.state[0] ** 2 + 0.1 * self.state[1] ** 2 + 0.01 * action ** 2)
        done = abs(self.state[0]) > 10
        return self.state.copy(), reward, done

def train_sac():
    """训练SAC"""
    agent = SACAgent(n_states=2, n_actions=1)
    buffer = ReplayBuffer(capacity=5000)
    env = ContinuousEnv()
    rewards_history = []
    
    for ep in range(100):
        s = env.reset()
        total = 0
        
        for _ in range(100):
            a = agent.get_action(s)
            ns, r, done = env.step(a)
            buffer.push((s, a, r, ns, done))
            s = ns
            total += r
            
            if len(buffer) >= 32:
                for _ in range(10):
                    batch = buffer.sample(32)
                    for s_, a_, r_, ns_, done_ in batch:
                        agent.update(s_, a_, r_, ns_, done_)
        rewards_history.append(total)
    
    print("SAC训练结果:")
    print(f"  最终回报: {rewards_history[-1]:.2f}")
    print(f"  平均回报(最后10局): {np.mean(rewards_history[-10:]):.2f}")

train_sac()

---

### 编程题2：实现自动调整熵温度的SAC

In [ ]:
class AutoAlphaSAC:
    """自动熵温度SAC"""
    def __init__(self, n_states, n_actions):
        self.actor_mean = np.zeros(n_actions)
        self.actor_log_std = np.zeros(n_actions)
        self.critic1 = np.zeros(n_actions)
        self.log_alpha = np.log(0.2)
        self.target_entropy = -n_actions
    
    def get_alpha(self): return np.exp(self.log_alpha)
    
    def update_alpha(self, entropy):
        alpha_loss = self.log_alpha * (entropy - self.target_entropy)
        self.log_alpha -= 0.001 * alpha_loss
    
    def update(self, s, a, entropy, q):
        self.update_alpha(entropy)
        alpha = self.get_alpha()
        actor_loss = -q - alpha * np.exp(self.actor_log_std)
        self.actor_mean -= 0.001 * actor_loss

def test_auto_alpha():
    """测试自动alpha"""
    agent = AutoAlphaSAC(n_states=2, n_actions=1)
    env = ContinuousEnv()
    alpha_history = []
    
    for ep in range(50):
        s = env.reset()
        entropy = 0.5
        q = 0.5
        agent.update(s, None, entropy, q)
        alpha_history.append(agent.get_alpha())
    
    print("自动熵温度SAC:")
    print(f"  初始alpha: {alpha_history[0]:.3f}")
    print(f"  最终alpha: {alpha_history[-1]:.3f}")

test_auto_alpha()

---

### 编程题3：实现SAC与DDPG的性能对比

In [ ]:
class SimpleDDPG:
    """简化DDPG"""
    def __init__(self, n_states):
        self.actor = np.random.randn(n_states) * 0.1
        self.critic = np.random.randn(n_states) * 0.1
    def get_action(self, s):
        action = np.tanh(np.dot(s, self.actor))
        return np.clip(action + np.random.randn() * 0.1, -1, 1)

def compare_sac_ddpg():
    """对比SAC与DDPG"""
    sac_agent = SACAgent(n_states=2, n_actions=1)
    ddpg_agent = SimpleDDPG(n_states=2)
    env = ContinuousEnv()
    buffer = ReplayBuffer(5000)
    
    sac_rewards, ddpg_rewards = [], []
    
    for agent, rewards in [(sac_agent, sac_rewards), (ddpg_agent, ddpg_rewards)]:
        for ep in range(50):
            s = env.reset()
            total = 0
            for _ in range(100):
                a = agent.get_action(s)
                ns, r, done = env.step(a)
                buffer.push((s, a, r, ns, done))
                s = ns
                total += r
                if done: break
            
            if len(buffer) >= 32:
                for _ in range(10):
                    batch = buffer.sample(32)
                    for s_, a_, r_, ns_, done_ in batch:
                        agent.update(s_, a_, r_, ns_, done_)
            rewards.append(total)
    
    print("SAC vs DDPG对比:")
    print(f"  SAC平均回报: {np.mean(sac_rewards):.2f}")
    print(f"  DDPG平均回报: {np.mean(ddpg_rewards):.2f}")

compare_sac_ddpg()

print("\n第17章 SAC算法 - 编程题完成!")